# 🧪 01 - Preparación, Limpieza y Validación de Datos (RotBot English Coach)

Este notebook procesa y valida el dataset de entrenamiento para **RotBot** (200 ejemplos con personalidad sarcástica, inteligente, amigable y enfoque de coaching en inglés sin emojis).

### Objetivos del Notebook:
1. **Carga Automática:** Detectar y parsear `data/raw/rotbot_training_dataset.sql` (200 registros estructurados).
2. **Análisis Exploratorio:** Examinar la distribución por categoría (`grammar_correction`, `conversation`, `idiom`, `roleplay`, `smalltalk`) y nivel (`beginner`, `intermediate`, `advanced`).
3. **Control de Calidad:** Validar integridad de mensajes, alternancia de roles y conteo de tokens.
4. **División Train / Val:** Generar split 80/20 (160 train / 40 val).
5. **Exportación JSONL:** Guardar en `data/processed/train.jsonl` y `data/processed/val.jsonl` y verificar.

In [ ]:
# 1. Configuración de entorno y rutas
import os
import sys
from pathlib import Path

# Añadir el directorio raíz al path para importar src
PROJECT_ROOT = Path(os.path.abspath("")).resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from src.parser import sql_to_dataframe, sql_to_jsonl, find_default_sql_file, DEFAULT_SYSTEM_PROMPT
from src.validator import validate_jsonl_dataset, print_dataset_summary

print(f"✅ Proyecto cargado en: {PROJECT_ROOT}")
print(f"📌 System Prompt Rotbot:\n{DEFAULT_SYSTEM_PROMPT}")

## 2. Detección y Carga del Dataset SQL
Identificamos el archivo SQL en `data/raw/` y lo cargamos a un DataFrame de Pandas.

In [ ]:
RAW_SQL_PATH = find_default_sql_file(str(PROJECT_ROOT))
print(f"📂 Archivo SQL detectado: {RAW_SQL_PATH}")

df = sql_to_dataframe(RAW_SQL_PATH)
print(f"Total de registros cargados: {len(df)}")
display(df.head(5) if not df.empty else "⚠️ No se encontraron registros")

## 3. Análisis de Distribución (Categorías y Niveles)
Verificamos la variedad de temas y niveles de dificultad presentes en el dataset.

In [ ]:
if 'category' in df.columns:
    print("--- 📂 Distribución por Categoría ---")
    print(df['category'].value_counts())
    
if 'level' in df.columns:
    print("\n--- 📊 Distribución por Nivel ---")
    print(df['level'].value_counts())
    
if 'notes' in df.columns:
    errors_targeted = df['notes'].dropna()
    print(f"\n--- 🎯 Ejemplos con errores específicos mapeados: {len(errors_targeted)} ---")
    print(errors_targeted.head(5))

## 4. Inspección de Muestras de la Personalidad de Rotbot
Examinamos ejemplos representativos del tono (sarcástico, ingenioso, llamando 'boss' al usuario, sin emojis).

In [ ]:
for i in [0, 8, 50, 100]:
    if i < len(df):
        row = df.iloc[i]
        print(f"="*60)
        print(f"Ejemplo #{i+1} [{row.get('category', 'N/A')} | {row.get('level', 'N/A')}]")
        print(f"👤 USER:      {row['user_message']}")
        print(f"🤖 ROTBOT:    {row['assistant_message']}")
        if pd.notna(row.get('notes')):
            print(f"🎯 ERROR MAP: {row['notes']}")

## 5. División y Exportación a JSONL (Train / Validation)
Dividimos los 200 ejemplos en 80% (160 ejemplos) para entrenamiento y 20% (40 ejemplos) para validación.

In [ ]:
TRAIN_JSONL_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "train.jsonl")
VAL_JSONL_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "val.jsonl")

train_count, val_count = sql_to_jsonl(
    sql_path=RAW_SQL_PATH,
    output_train_path=TRAIN_JSONL_PATH,
    output_val_path=VAL_JSONL_PATH,
    val_ratio=0.2,
    format_type="chatml",
    seed=42
)

print(f"✅ Exportación completada con éxito:")
print(f"  - 🏋️ Train: {train_count} ejemplos -> {TRAIN_JSONL_PATH}")
print(f"  - 🧪 Val:   {val_count} ejemplos -> {VAL_JSONL_PATH}")

## 6. Validación de Calidad y Estadísticas de Tokens
Verificamos que todos los registros cumplan con el esquema ChatML y calculamos las métricas de tokens.

In [ ]:
train_report = validate_jsonl_dataset(TRAIN_JSONL_PATH)
print_dataset_summary(train_report, title="Reporte de Validación - Train Dataset")

val_report = validate_jsonl_dataset(VAL_JSONL_PATH)
print_dataset_summary(val_report, title="Reporte de Validación - Validation Dataset")